In [ ]:
import torch
import subprocess

print("🔍 GPU Status:")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}GB")
    print(f"PyTorch: {torch.__version__}")
else:
    print("⚠️ Go to Runtime → Change Runtime Type and select GPU")
    
print("\n📦 Checking Python...")
print(f"Python version: {subprocess.check_output(['python', '--version']).decode().strip()}")

## ✅ Step 0: GPU Setup & Verify

# 🚀 RL Training on Google Colab
## Train Policy with PPO on CyberMultiAgentEnv

This notebook trains a reinforcement learning policy on your custom environment with real-time rewards tracking.

## ✅ Step 1: Mount Drive & Clone Repository

In [ ]:
from google.colab import drive
import os
import subprocess

# Mount Google Drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

# Clone repository
REPO_URL = "https://github.com/YOUR_USERNAME/MetaHackUI.git"  # 👈 UPDATE THIS

if not os.path.exists("/content/MetaHackUI"):
    print("📦 Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, "/content/MetaHackUI"], check=True)
    os.chdir("/content/MetaHackUI")
else:
    os.chdir("/content/MetaHackUI")

print(f"✅ Working directory: {os.getcwd()}")
print(f"📂 Project structure check:")
for folder in ['agents', 'openenv', 'rl', 'dataset']:
    exists = "✅" if os.path.exists(folder) else "❌"
    print(f"   {exists} {folder}/")

## ✅ Step 2: Install Dependencies

In [ ]:
import subprocess
import sys

print("📦 Installing dependencies...")

packages = [
    # Core
    "pydantic>=2.0.0",
    "python-dotenv",
    "aiohttp",
    "numpy",
    "scipy",
    # LLM
    "openai",
    "langchain",
    "langchain-openai",
    "langchain-community",
    "langchain-core",
    # Embeddings
    "sentence-transformers",
    "faiss-cpu",
    # RL
    "gymnasium>=0.29.0",
    "stable-baselines3>=2.0.0",
    "torch>=2.0.0",
    # Viz
    "matplotlib",
    "seaborn",
    "pandas",
    "tensorboard",
    "rich"
]

for package in packages:
    print(f"  Installing {package.split('>=')[0]}...", end=" ")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    print("✅")

print("\n✅ All dependencies installed!")

## ✅ Step 3: Load Training Data & Cases

In [ ]:
import json
import os
from pathlib import Path

print("📂 Loading incident cases...")

# Load cases dataset
cases_path = "/content/MetaHackUI/dataset/cases.json"
if os.path.exists(cases_path):
    with open(cases_path, 'r') as f:
        cases = json.load(f)
    print(f"✅ Loaded {len(cases)} incident cases")
    
    # Display sample case structure
    if cases:
        sample = cases[0]
        print(f"\n📋 Sample case structure:")
        print(f"   Keys: {list(sample.keys())}")
        if 'logs' in sample:
            print(f"   Logs: {len(sample['logs'])} entries")
        if 'requirements' in sample:
            print(f"   Requirements: {sample['requirements'][:2]}...")
else:
    print(f"⚠️ {cases_path} not found!")
    print("   Make sure dataset/cases.json is synced from GitHub")
    cases = []  # Empty for now

## ✅ Step 4: Initialize Agents & Evaluator

In [ ]:
import sys
sys.path.insert(0, '/content/MetaHackUI')

from agents import CodeAgent, CriticAgent, FusionAgent, LogAgent, ReqAgent
from evaluator import Evaluator
from schemas import PolicyState

print("🤖 Initializing agents...")

# Initialize agents (these will use mock/placeholder implementations)
agents_dict = {
    "log_agent": LogAgent(),
    "code_agent": CodeAgent(),
    "req_agent": ReqAgent(),
    "fusion_agent": FusionAgent(),
    "critic_agent": CriticAgent(),
}
print("✅ Agents initialized")

# Initialize evaluator
evaluator = Evaluator()
print("✅ Evaluator initialized")

# Initialize policy state
policy = PolicyState(
    log_sensitivity=0.5,
    code_sensitivity=0.5,
    req_sensitivity=0.5,
    fusion_temperature=0.7,
    confidence_threshold=0.5
)
print(f"✅ Policy state initialized: {policy}")

print("\n📊 Component Summary:")
print(f"   Agents: {len(agents_dict)}")
print(f"   Cases: {len(cases)}")
print(f"   Policy parameters: 5 (log_sens, code_sens, req_sens, fusion_temp, conf_thresh)")

## ✅ Step 5: Create Training Environment

In [ ]:
from openenv import CyberMultiAgentEnv

print("🌍 Creating training environment...")

# Create environment
env = CyberMultiAgentEnv(
    cases=cases[:50] if cases else [],  # Use first 50 cases for training
    agents_dict=agents_dict,
    evaluator=evaluator,
    policy_state=policy
)

print("✅ Environment created")

# Test environment
print("\n🧪 Testing environment...")
try:
    obs, info = env.reset()
    print(f"✅ Environment reset successful")
    print(f"   Observation shape: {obs.shape}")
    print(f"   Observation space: {env.observation_space}")
    print(f"   Action space: {env.action_space}")
    
    # Test a random action
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"✅ Step executed successfully")
    print(f"   Reward: {reward:.4f}")
    print(f"   Terminated: {terminated}")
    
except Exception as e:
    print(f"❌ Environment error: {e}")
    print("   This might be due to missing mock data, which is OK for this demo")

## ✅ Step 6: Setup PPO Training with Callbacks

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
import numpy as np
import os

print("⚙️  Setting up PPO training...")

# Create output directory
output_dir = "/content/drive/My Drive/MetaHackUI_RL_results"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f"{output_dir}/checkpoints", exist_ok=True)
print(f"✅ Output directory: {output_dir}")

# Custom callback for monitoring
class MetricsCallback(BaseCallback):
    def __init__(self, log_dir="/content/drive/My Drive/MetaHackUI_RL_results"):
        super().__init__()
        self.log_dir = log_dir
        self.episode_rewards = []
        self.episode_lengths = []
        self.current_episode_reward = 0
        self.current_episode_length = 0
        
    def _on_step(self) -> bool:
        # Track rewards
        self.current_episode_reward += self.locals["rewards"][0]
        self.current_episode_length += 1
        
        # Episode ends
        if self.locals["dones"][0]:
            self.episode_rewards.append(self.current_episode_reward)
            self.episode_lengths.append(self.current_episode_length)
            
            if len(self.episode_rewards) % 10 == 0:
                mean_reward = np.mean(self.episode_rewards[-10:])
                print(f"  Episode {len(self.episode_rewards)}: reward={self.current_episode_reward:.4f}, mean(last 10)={mean_reward:.4f}")
            
            self.current_episode_reward = 0
            self.current_episode_length = 0
        
        return True

callback = MetricsCallback(log_dir=output_dir)
print("✅ Callback configured")

# Setup PPO model
print("\n🤖 Creating PPO model...")
model = PPO(
    "MlpPolicy",
    env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    verbose=1,
    tensorboard_log=f"{output_dir}/tb_logs",
)
print("✅ PPO model created")
print(f"   Total timesteps to train: 10000")
print(f"   Steps per epoch: 2048")
print(f"   Expected episodes: ~10000 (depending on case complexity)")

## ✅ Step 7: Train PPO Model with Real-Time Monitoring

In [ ]:
print("🚀 Starting PPO training...")
print("⏱️  ETA: ~5-10 minutes for 10000 timesteps")
print("💡 Check TensorBoard below to monitor in real-time\n")

try:
    # Train the model
    model.learn(
        total_timesteps=10000,  # Reduced for Colab time limits
        callback=callback,
        log_interval=100,
        progress_bar=True
    )
    
    print("\n✅ Training complete!")
    print(f"   Total episodes: {len(callback.episode_rewards)}")
    print(f"   Mean reward: {np.mean(callback.episode_rewards):.4f}")
    print(f"   Max reward: {np.max(callback.episode_rewards):.4f}")
    print(f"   Min reward: {np.min(callback.episode_rewards):.4f}")
    
except KeyboardInterrupt:
    print("\n⚠️  Training interrupted")
except Exception as e:
    print(f"\n❌ Training error: {e}")
    print("   Check GPU memory or environment setup")

## ✅ Step 8: Save Model & Checkpoints

In [ ]:
import os

output_dir = "/content/drive/My Drive/MetaHackUI_RL_results"
model_path = f"{output_dir}/ppo_policy"

print("💾 Saving trained model...")
model.save(model_path)
print(f"✅ Model saved: {model_path}.zip")

# Save training metrics
import json
import numpy as np

metrics = {
    "model_type": "PPO",
    "total_timesteps": 10000,
    "episodes_trained": len(callback.episode_rewards),
    "mean_episode_reward": float(np.mean(callback.episode_rewards)) if callback.episode_rewards else 0,
    "max_episode_reward": float(np.max(callback.episode_rewards)) if callback.episode_rewards else 0,
    "min_episode_reward": float(np.min(callback.episode_rewards)) if callback.episode_rewards else 0,
    "final_policy_params": {
        "learning_rate": 3e-4,
        "n_steps": 2048,
        "batch_size": 64,
        "n_epochs": 10,
        "gamma": 0.99,
        "gae_lambda": 0.95,
        "clip_range": 0.2
    }
}

metrics_path = f"{output_dir}/training_metrics.json"
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"✅ Metrics saved: {metrics_path}")

## ✅ Step 9: Generate Training Graphs

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import uniform_filter1d

output_dir = "/content/drive/My Drive/MetaHackUI_RL_results"

print("📊 Generating training graphs...")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("RL Training Metrics - PPO on CyberMultiAgentEnv", fontsize=16, fontweight='bold')

# Plot 1: Episode Rewards
ax = axes[0, 0]
if callback.episode_rewards:
    episodes = np.arange(len(callback.episode_rewards))
    ax.plot(episodes, callback.episode_rewards, 'b-', alpha=0.5, linewidth=1, label='Reward')
    
    # Add moving average
    if len(callback.episode_rewards) > 10:
        moving_avg = uniform_filter1d(callback.episode_rewards, size=10)
        ax.plot(episodes, moving_avg, 'r-', linewidth=2.5, label='Moving Avg (10)')
    
    ax.set_title("Episode Rewards", fontweight='bold', fontsize=12)
    ax.set_xlabel("Episode")
    ax.set_ylabel("Reward")
    ax.grid(True, alpha=0.3)
    ax.legend()
else:
    ax.text(0.5, 0.5, "No rewards recorded yet", ha='center', va='center')

# Plot 2: Episode Length Distribution
ax = axes[0, 1]
if callback.episode_lengths:
    ax.hist(callback.episode_lengths, bins=20, color='green', alpha=0.7, edgecolor='black')
    ax.set_title("Episode Length Distribution", fontweight='bold', fontsize=12)
    ax.set_xlabel("Steps per Episode")
    ax.set_ylabel("Frequency")
    ax.grid(True, alpha=0.3, axis='y')
else:
    ax.text(0.5, 0.5, "No episodes recorded yet", ha='center', va='center')

# Plot 3: Cumulative Reward
ax = axes[1, 0]
if callback.episode_rewards:
    cumulative_reward = np.cumsum(callback.episode_rewards)
    ax.plot(cumulative_reward, 'g-', linewidth=2)
    ax.fill_between(np.arange(len(cumulative_reward)), cumulative_reward, alpha=0.3, color='green')
    ax.set_title("Cumulative Reward Over Episodes", fontweight='bold', fontsize=12)
    ax.set_xlabel("Episode")
    ax.set_ylabel("Cumulative Reward")
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, "No cumulative data yet", ha='center', va='center')

# Plot 4: Training Summary (Text)
ax = axes[1, 1]
ax.axis('off')

summary_text = f"""
Training Configuration:
━━━━━━━━━━━━━━━━━━━━━━━━━━━
Algorithm: PPO (Proximal Policy Optimization)
Learning Rate: 3e-4
N Steps (batch): 2048
Batch Size: 64
Epochs per batch: 10
Gamma (discount): 0.99
GAE Lambda: 0.95
Clip Range: 0.2

Results:
━━━━━━━━━━━━━━━━━━━━━━━━━━━
Total Episodes: {len(callback.episode_rewards)}
Mean Reward: {np.mean(callback.episode_rewards):.4f if callback.episode_rewards else 'N/A'}
Max Reward: {np.max(callback.episode_rewards):.4f if callback.episode_rewards else 'N/A'}
Min Reward: {np.min(callback.episode_rewards):.4f if callback.episode_rewards else 'N/A'}
Avg Episode Length: {np.mean(callback.episode_lengths):.1f} steps if callback.episode_lengths else 'N/A'}

Artifacts:
━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ Model: {output_dir}/ppo_policy.zip
✅ Metrics: {output_dir}/training_metrics.json
✅ TensorBoard: {output_dir}/tb_logs
"""

ax.text(0.05, 0.95, summary_text, fontfamily='monospace', fontsize=9.5,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))

plt.tight_layout()
graph_path = f"{output_dir}/training_graphs.png"
plt.savefig(graph_path, dpi=150, bbox_inches='tight')
print(f"✅ Graphs saved: {graph_path}")
plt.show()

## ✅ Step 10: View TensorBoard Logs

In [ ]:
output_dir = "/content/drive/My Drive/MetaHackUI_RL_results"
tb_log_dir = f"{output_dir}/tb_logs"

print("📈 Loading TensorBoard...")
print(f"   Log directory: {tb_log_dir}\n")

try:
    %load_ext tensorboard
    %tensorboard --logdir {tb_log_dir}
except Exception as e:
    print(f"Note: {e}")
    print("TensorBoard logs are still saved to Drive for later viewing")

## ✅ OPTIONAL: Evaluate Trained Policy

In [ ]:
from stable_baselines3 import PPO
import numpy as np

output_dir = "/content/drive/My Drive/MetaHackUI_RL_results"
model_path = f"{output_dir}/ppo_policy"

print("🧪 Evaluating trained policy...")

# Load trained model
model = PPO.load(model_path, env=env)

# Run evaluation episodes
n_eval_episodes = 5
eval_rewards = []
eval_lengths = []

print(f"\nRunning {n_eval_episodes} evaluation episodes...\n")

for episode in range(n_eval_episodes):
    obs, _ = env.reset()
    episode_reward = 0
    episode_length = 0
    done = False
    
    while not done:
        # Use deterministic policy (no exploration noise)
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        
        episode_reward += reward
        episode_length += 1
        done = terminated or truncated
    
    eval_rewards.append(episode_reward)
    eval_lengths.append(episode_length)
    
    print(f"  Episode {episode + 1}: reward={episode_reward:.4f}, length={episode_length}")

print(f"\n📊 Evaluation Results:")
print(f"   Mean Reward: {np.mean(eval_rewards):.4f}")
print(f"   Std Reward: {np.std(eval_rewards):.4f}")
print(f"   Mean Episode Length: {np.mean(eval_lengths):.1f}")
print(f"✅ Evaluation complete!")

## 🎯 Summary & Next Steps\n\n### ✅ Completed:\n- ✅ Trained PPO policy on CyberMultiAgentEnv\n- ✅ Generated reward curves and training metrics\n- ✅ Saved trained model to Google Drive\n- ✅ Evaluated policy performance\n\n### 📥 Download Results from Drive:\n```\nMetaHackUI_RL_results/\n├── ppo_policy.zip           # Trained model\n├── training_graphs.png      # Metrics visualization\n├── training_metrics.json    # Numerical results\n└── tb_logs/                 # TensorBoard logs\n```\n\n### 🔄 Next Steps:\n1. **Combine**: Fine-tuned SFT model + trained RL policy\n2. **Deploy**: Integrate into detection pipeline\n3. **Iterate**: Run more episodes, adjust hyperparameters\n4. **Benchmark**: Compare vs baseline policies"